# 🐶 POC: Fast Prompting para Catálogo de Refugio de Animales

**Proyecto:** Huellas con Futuro
**Autora:** Sofía Algamiz
**Objetivo:** ejecutar una solución de *Fast Prompting* en **una sola llamada a la API** para generar, a partir de una ficha de rescate, la biografía de catálogo, el copy para redes sociales, la ficha resumen y el prompt fotográfico calibrado.

> Si no hay una `OPENAI_API_KEY` configurada, la notebook corre en **modo demo offline**: genera una salida de ejemplo con los mismos datos para poder mostrar el funcionamiento sin necesidad de credenciales ni de gastar consultas reales.

In [ ]:
# Instalación de dependencias (si hace falta)
# !pip install -r requirements.txt

import os
import json
import html
import ipywidgets as widgets
from IPython.display import display, HTML
from openai import OpenAI

# La API key se lee de la variable de entorno OPENAI_API_KEY.
# Si no está seteada, se usa una key ficticia para poder instanciar el
# cliente igual: la llamada real va a fallar y el except de
# generar_perfil_adopcion() activa el modo demo (ver Celda 4).
API_KEY = os.environ.get("OPENAI_API_KEY")
MODO_DEMO = API_KEY is None

if MODO_DEMO:
    print("⚠️ No se encontró OPENAI_API_KEY en el entorno: la notebook va a correr en modo demo offline.")

client = OpenAI(api_key=API_KEY or "sk-demo-mode")

## Prompt maestro: System Instructions + Few-Shot + salida JSON forzada

In [ ]:
# PROMPT MAESTRO: combina System Instructions + Few-Shot Prompting + Structured Output (JSON)
SYSTEM_PROMPT = """Eres un redactor experto en marketing de adopción responsable para protectoras de animales.
Tu labor es transformar fichas técnicas de rescate en perfiles atractivos y éticos, maximizando la conexión
emocional sin mentir ni omitir datos de salud o de conducta.

Debes responder SIEMPRE con un único objeto JSON válido con esta estructura exacta:
{
  "titulo_catalogo": "Título empático y gancho (máximo 6 palabras)",
  "biografia_narrativa": "Biografía de 2-3 párrafos enfocada en su resiliencia, personalidad y su compañero ideal.",
  "post_redes": "Copy para Instagram en primera o tercera persona, con emojis y 3 hashtags.",
  "ficha_resumen": {
    "edad": "...",
    "tamano": "...",
    "compatibilidad": "...",
    "salud": "..."
  },
  "prompt_t2i": "Prompt en INGLÉS para generación de imagen (Nightcafe / Leonardo AI / DALL-E 3), con lente, iluminación cálida y los rasgos físicos EXACTOS del animal."
}"""

# Ejemplo few-shot: fija el estilo y el nivel de detalle esperado
FEW_SHOT_EXAMPLE_INPUT = (
    "Nombre: Milo | Especie: Perro mestizo | Edad: 4 años | Tamaño: Grande | "
    "Color: Negro azabache con pecho blanco | Orejas: Caídas | "
    "Personalidad: Muy tranquilo, dormilón, le teme a las tormentas | "
    "Salud: Castrado, displasia leve en cadera."
)

FEW_SHOT_EXAMPLE_OUTPUT = {
    "titulo_catalogo": "Milo: Un gigante de corazón noble",
    "biografia_narrativa": (
        "Milo demuestra que el tamaño de un perro es igual al tamaño de su amor. "
        "Tras superar un pasado de abandono, hoy busca un rincón cómodo donde descansar.\n\n"
        "Es un compañero extremadamente tranquilo cuyo pasatiempo favorito son las siestas al sol. "
        "Requiere paseos suaves debido a su displasia y un hogar comprensivo que le brinde "
        "seguridad los días de tormenta."
    ),
    "post_redes": (
        "🐾 ¡Hola! Soy Milo, un osito negro de 4 años buscando un sofá donde darte todo mi amor. "
        "💤 Me encantan las tardes de tranquilidad. ¿Me das una oportunidad? "
        "#AdoptaNoCompres #MiloBuscaHogar #AdoptaUnNegrito"
    ),
    "ficha_resumen": {
        "edad": "4 años",
        "tamano": "Grande",
        "compatibilidad": "Ideal familias tranquilas",
        "salud": "Castrado, displasia leve controlada",
    },
    "prompt_t2i": (
        "A professional studio photo of a large mixed-breed dog, short glossy jet-black coat with a "
        "distinct white patch on chest, gentle expressive brown eyes, floppy ears, relaxed lying posture "
        "on a soft woolen rug in a sunlit living room. Warm ambient lighting, indoor plants in blurred "
        "background, 85mm portrait lens, f/2.0, photorealistic, 8k resolution."
    ),
}

## Función de generación (Fast Prompting: una sola llamada a la API)

In [ ]:
def generar_perfil_adopcion(nombre, especie, edad, tamano, rasgos_fisicos, personalidad, salud_historial):
    """
    Ejecuta UNA ÚNICA llamada a la API (Fast Prompting) para generar, de una sola vez,
    el título, la biografía, el copy de redes, la ficha resumen y el prompt de imagen.
    """
    user_input = (
        f"Genera el perfil para el siguiente animal rescatado:\n"
        f"- Nombre: {nombre}\n"
        f"- Especie: {especie}\n"
        f"- Edad: {edad}\n"
        f"- Tamaño: {tamano}\n"
        f"- Rasgos Físicos Distintivos: {rasgos_fisicos}\n"
        f"- Personalidad: {personalidad}\n"
        f"- Salud e Historial: {salud_historial}"
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Ejemplo:\n{FEW_SHOT_EXAMPLE_INPUT}"},
        {"role": "assistant", "content": json.dumps(FEW_SHOT_EXAMPLE_OUTPUT, ensure_ascii=False)},
        {"role": "user", "content": user_input},
    ]

    if MODO_DEMO:
        return _perfil_demo(nombre, especie, edad, tamano, rasgos_fisicos, personalidad, salud_historial)

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",  # modelo rápido y de costo muy bajo
            messages=messages,
            response_format={"type": "json_object"},
            temperature=0.7,
            max_tokens=800,
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"⚠️ No se pudo consultar la API ({e}). Muestro un resultado de demo en su lugar.")
        return _perfil_demo(nombre, especie, edad, tamano, rasgos_fisicos, personalidad, salud_historial)


def _perfil_demo(nombre, especie, edad, tamano, rasgos_fisicos, personalidad, salud_historial):
    """Salida de ejemplo para poder mostrar la POC sin consumir la API real."""
    hashtag_nombre = nombre.replace(" ", "")
    return {
        "titulo_catalogo": f"{nombre}: listo para llenar tu vida de alegría",
        "biografia_narrativa": (
            f"{nombre} es un rescatado especial de {edad}. Destaca por ser {personalidad}. "
            f"Está buscando una familia responsable que le brinde los cuidados que merece tras su recuperación."
        ),
        "post_redes": (
            f"Conoce a {nombre} ({edad}). Es {personalidad}. ¡Escribinos para adoptarlo! "
            f"#Adopta{hashtag_nombre} #AdopcionResponsable #DaleUnHogar"
        ),
        "ficha_resumen": {
            "edad": edad,
            "tamano": tamano,
            "compatibilidad": "A evaluar con la familia",
            "salud": salud_historial,
        },
        "prompt_t2i": (
            f"Professional photography portrait of a {tamano.lower()} {especie.lower()}, {rasgos_fisicos}, "
            f"sitting in a warm bright living room, photorealistic, 85mm lens, natural light."
        ),
    }

## Interfaz interactiva

In [ ]:
# Widgets de carga de datos
txt_nombre = widgets.Text(value="Luna", description="Nombre:")
txt_especie = widgets.Dropdown(options=["Perro", "Gato"], value="Perro", description="Especie:")
txt_edad = widgets.Text(value="3 años", description="Edad:")
txt_tamano = widgets.Dropdown(options=["Pequeño", "Mediano", "Grande"], value="Mediano", description="Tamaño:")
txt_rasgos = widgets.Text(
    value="Pelaje corto canela, ojos miel, oreja derecha doblada",
    description="Rasgos:",
    style={"description_width": "initial"},
)
txt_personalidad = widgets.Text(
    value="Sociable, cariñosa, activa, se lleva bien con gatos",
    description="Carácter:",
    style={"description_width": "initial"},
)
txt_salud = widgets.Text(value="Vacunada, castrada, sana", description="Salud:")

btn_generar = widgets.Button(
    description="🐾 Generar Perfil Completo",
    button_style="success",
    layout=widgets.Layout(width="50%"),
)
out_resultado = widgets.Output()


def on_button_clicked(b):
    with out_resultado:
        out_resultado.clear_output()
        print("⏳ Procesando mediante Fast Prompting (1 sola llamada)...")
        data = generar_perfil_adopcion(
            txt_nombre.value,
            txt_especie.value,
            txt_edad.value,
            txt_tamano.value,
            txt_rasgos.value,
            txt_personalidad.value,
            txt_salud.value,
        )

        # Se escapa todo el contenido generado antes de insertarlo en HTML,
        # para que caracteres como <, > o & no rompan el renderizado.
        titulo = html.escape(data["titulo_catalogo"])
        bio = html.escape(data["biografia_narrativa"]).replace("\n", "<br>")
        post = html.escape(data["post_redes"])
        prompt_t2i = html.escape(data["prompt_t2i"])
        ficha = data["ficha_resumen"]

        display(HTML(f"""
        <div style="border: 2px solid #2a5298; border-radius: 8px; padding: 15px; background: #fdfdfd; margin-top: 10px;">
            <h2 style="color: #1e3c72; margin-top:0;">{titulo}</h2>

            <h4 style="color: #e74c3c;">📖 Biografía para catálogo:</h4>
            <p style="line-height: 1.5;">{bio}</p>

            <h4 style="color: #27ae60;">📱 Copy para redes sociales:</h4>
            <div style="background: #f1f2f6; padding: 10px; border-radius: 5px; font-style: italic;">{post}</div>

            <h4 style="color: #8e44ad;">📋 Ficha resumen rápida:</h4>
            <ul>
                <li><strong>Edad:</strong> {html.escape(str(ficha.get("edad", "")))}</li>
                <li><strong>Tamaño:</strong> {html.escape(str(ficha.get("tamano", "")))}</li>
                <li><strong>Compatibilidad:</strong> {html.escape(str(ficha.get("compatibilidad", "")))}</li>
                <li><strong>Salud:</strong> {html.escape(str(ficha.get("salud", "")))}</li>
            </ul>

            <h4 style="color: #d35400;">🎨 Prompt T2I generado (calibrado a los rasgos reales):</h4>
            <code style="background: #2f3542; color: #f1f2f6; padding: 8px; display: block; border-radius: 4px;">{prompt_t2i}</code>
            <p style="font-size: 0.85em; color: #888; margin-top: 8px;">
                Copiá este prompt en Nightcafe, Leonardo AI o DALL-E 3 para generar la imagen conceptual.
            </p>
        </div>
        """))


btn_generar.on_click(on_button_clicked)

display(HTML("<h3>📋 Formulario de carga del rescatado</h3>"))
display(widgets.VBox([
    txt_nombre, txt_especie, txt_edad, txt_tamano,
    txt_rasgos, txt_personalidad, txt_salud,
    btn_generar, out_resultado,
]))

## Conclusión: ¿mejora el Fast Prompting la propuesta de la Preentrega 1?

Sí. En la Preentrega 1 cada resultado (biografía emotiva, biografía en primera persona, prompt de imagen
canino, prompt de imagen felino) se generaba con **prompts independientes**, ejecutados manualmente uno por
uno. Acá, el prompt maestro con *few-shot* + salida JSON forzada permite obtener título, biografía, copy de
redes y prompt de imagen **en una sola llamada a la API**, lo que:

- reduce el costo por animal procesado a menos de USD 0,001,
- reduce la latencia total (1 consulta en vez de 3-4),
- y simplifica la reutilización: un voluntario solo completa un formulario, no necesita elegir ni combinar
  varios prompts distintos.